# 06 · Public spending registers

Research log: [Step 7](../docs/research_log.md#step-7--registre-publice-de-cheltuieli) (in Romanian). Data and checks as of October 2026.

**Question.** Do public registers of payments to suppliers publish the supplier's VAT number?

**Method.** Two local authorities, same month (June 2026): every 9-digit number in every cell tested with the checksum; columns and redactions inspected. The HM Treasury £25k guidance was read in the PDF (not in this notebook).

**Result (from the research log).**
- Dorset 11,113 rows (52% redacted) and Kirklees 5,669 rows: 0 VAT numbers in 16,782 rows.
- The 6 checksum-passing numbers at Kirklees are not VAT numbers (4 code artefacts, 2 reference numbers).
- Central government: supplier VAT number is collected but "not to be published until requested by the Treasury".

**Notes for re-running.**
- `glob('../data/registre/*')` now also matches the Find a Tender files in that folder (they were added later).

*Cells are kept as they were run, outputs included; only file paths were updated to the new `data/` layout and session timestamps in outputs were cut to the month.*

In [4]:
import re, glob
import pandas as pd

def checksum_ok(nr9):
    if len(nr9) != 9 or not nr9.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr9[:7], range(8, 1, -1))) + int(nr9[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

for cale in glob.glob("../data/registre/*"):
    df = (pd.read_excel(cale, dtype=str) if cale.endswith(("xlsx", "xls"))
          else pd.read_csv(cale, dtype=str, encoding="latin-1", on_bad_lines="skip"))
    print(f"\n{cale.split('/')[-1]}: {len(df)} rânduri")
    print("Coloane:", df.columns.tolist())
    # caută orice număr care arată ca TVA UK, în orice coloană
    text = " ".join(df.astype(str).fillna("").values.ravel())
    cand = {re.sub(r"\D", "", m) for m in re.findall(r"(?:GB)?\s?\d{3}\s?\d{4}\s?\d{2}(?!\d)", text)}
    valide = [c for c in cand if checksum_ok(c)]
    print("Numere de 9 cifre:", len(cand), "| trec checksum:", len(valide), valide[:5])

In [1]:
import re, glob
import pandas as pd

def checksum_ok(nr9):
    if len(nr9) != 9 or not nr9.isdigit():
        return False
    s = sum(int(d) * w for d, w in zip(nr9[:7], range(8, 1, -1))) + int(nr9[7:])
    return s % 97 == 0 or (s + 55) % 97 == 0

fisiere = glob.glob("../data/registre/*")
print("Fișiere găsite:", fisiere)

for cale in fisiere:
    df = (pd.read_excel(cale, dtype=str) if cale.endswith(("xlsx", "xls"))
          else pd.read_csv(cale, dtype=str, encoding="latin-1", on_bad_lines="skip"))
    print(f"\n{cale.split('/')[-1]}: {len(df)} rânduri")
    print("Coloane:", df.columns.tolist())

    text = " ".join(df.fillna("").astype(str).values.ravel())
    cand = {re.sub(r"\D", "", m) for m in
            re.findall(r"(?<!\d)(?:GB)?\s?\d{3}\s?\d{4}\s?\d{2}(?!\d)", text)}
    valide = [c for c in cand if checksum_ok(c)]
    print("Numere de 9 cifre:", len(cand), "| trec checksum:", len(valide), valide[:5])

    furnizori = df.iloc[:, [i for i, c in enumerate(df.columns)
                            if re.search(r"vendor|supplier|beneficiar|payee", str(c), re.I)]]
    if not furnizori.empty:
        col = furnizori.columns[0]
        redactate = df[col].str.contains("redact", case=False, na=False).sum()
        print(f"Furnizori unici: {df[col].nunique()} | rânduri redactate: {redactate}")

Fișiere găsite: ['../data/registre/dorset_2026_06.csv', '../data/registre/kirklees_2026_06.xlsx']

dorset_2026_06.csv: 11113 rânduri
Coloane: ['Directorate', 'Description', 'Invoice Number', 'Vendor', 'Date', 'Amount']
Numere de 9 cifre: 0 | trec checksum: 0 []
Furnizori unici: 1296 | rânduri redactate: 5776

kirklees_2026_06.xlsx: 5669 rânduri
Coloane: ['Payment Date', 'Transaction Number', 'Amount Excluding VAT', 'Vendor Number', 'Vendor Name', 'Company Number', 'Debit/Credit Indicator', 'PO Number', 'Document Type', 'Cost Centre', 'Cost Centre Description', 'Proclass Code', 'Proclass Description', 'Purpose of Spend']
Numere de 9 cifre: 437 | trec checksum: 6 ['912450791', '675341131', '600340606', '780420054', '200115687']
Furnizori unici: 1530 | rânduri redactate: 0


In [2]:
k = pd.read_excel("../data/registre/kirklees_2026_06.xlsx", dtype=str)

# 1. Din ce coloane vin cele 6 numere care trec checksum-ul
print("1. Unde apar numerele care trec checksum-ul:")
for nr in valide:
    for col in k.columns:
        if k[col].fillna("").str.replace(r"\D", "", regex=True).str.contains(nr).any():
            print(f"   {nr} -> coloana '{col}'")

# 2. Furnizori și rânduri redactate, după numele furnizorului
print("\n2. Vendor Name:")
print("   Furnizori unici:", k["Vendor Name"].nunique())
red = k["Vendor Name"].str.contains("redact", case=False, na=False)
print("   Rânduri redactate:", red.sum(), "din", len(k))

# 3. Cât de des e completat Company Number
cn = k["Company Number"].fillna("").str.strip()
print("\n3. Company Number completat:", (cn != "").sum(), "din", len(k), "rânduri")
furn = k.drop_duplicates("Vendor Name")
cn_f = furn["Company Number"].fillna("").str.strip()
print("   Furnizori unici cu Company Number:", (cn_f != "").sum(), "din", len(furn))
print("\n   Exemple:")
print(furn.loc[cn_f != "", ["Vendor Name", "Company Number"]].head(10).to_string(index=False))

1. Unde apar numerele care trec checksum-ul:
   200115687 -> coloana 'Purpose of Spend'
   260005808 -> coloana 'Purpose of Spend'

2. Vendor Name:
   Furnizori unici: 1219
   Rânduri redactate: 953 din 5669

3. Company Number completat: 1704 din 5669 rânduri
   Furnizori unici cu Company Number: 396 din 1219

   Exemple:
                       Vendor Name Company Number
Yorkshire Building Contractors Ltd        9455664
           British Gas Trading Ltd       03078711
                   Wm Projects Ltd       06627089
                     POLLOCK LIFTS          18971
                   1St Ability Ltd       06853960
                  B&K Passmore Ltd       04164081
     DawsonGroup Bus And Coach Ltd       02434773
            Northgate Vehicle Hire       01434157
      Kirklees School Services Ltd       04011161
       Portable Offices (Hire) Ltd        4201828


In [3]:
pd.set_option("display.max_colwidth", 300)
for nr in ["200115687", "260005808"]:
    rand = k[k["Purpose of Spend"].fillna("").str.replace(r"\D", "", regex=True).str.contains(nr)]
    print(rand[["Vendor Name", "Company Number", "Purpose of Spend"]].to_string(index=False), "\n")

          Vendor Name Company Number                         Purpose of Spend
Dewsbury Cleaning Ltd            NaN 26/27 - MDT (200115687) - DEWSBURY CLEAN 

             Vendor Name Company Number               Purpose of Spend
Croft Care Homes Limited        3505566 PN 260005808 20th - 31st March
Croft Care Homes Limited        3505566    PN 260005808 1 - 30th April 

